<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Taller: La Heladería de Villa de Leyva 🍦🧸
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Taller Práctico Evaluativo · Módulo 06
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Estadistica%20analisis%20y%20representacion%20de%20datos/homeworks/Para%20Dummies/06_Modelamiento_Estadistico_Hands_On_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## 🎈 ¿Qué vamos a hacer en este taller?

Doña Marta tiene una **heladería** en Villa de Leyva y quiere que un modelo prediga cuántos helados vende. Pero un modelo solo entiende **números**, se puede **memorizar** el pasado sin aprender nada y a veces **predice cosas imposibles**. Vas a:

1. 🔤 **Traducir las categorías a números sin inventar un orden:** numerar los puntos de venta (0, 1, 2, 3) le hace creer al modelo que el terminal está «después» de la plaza; las casillas de sí/no (*one-hot*) no.
2. 📝 **Poner a prueba el modelo con un examen sorpresa:** el estudiante que memoriza el simulacro saca 10 en el simulacro y 0 en el examen. Verás cómo unas columnas de lotería lo logran, y cómo cinco exámenes (validación cruzada) dicen la verdad.
3. 🔢 **Contar sin números negativos:** una recta puede predecir ¡−5 helados!; un modelo de conteos, nunca.
4. ✍️ **Contar con tus palabras** lo que aprendiste.

> 🧸 **Cómo funciona:** cada reto trae un cuadro de código con huecos `...` y pistas 💡. Reemplaza los `...` por lo que corresponda. Debajo hay una **autoverificación** que te orienta con cariño. No traen soluciones: ¡inténtalo tú!

### 📋 Instrucciones

- Completa solo los huecos `...`; no cambies las celdas de preparación ni de autoverificación.
- Al final usa *Restart & Run All*: todo debe correr sin errores y sin `...` pendientes.
- Todo funciona sin internet (datos inventados). Repasa si quieres: [El traductor de la plaza](../../06%20-%20Modelamiento%20Estadistico/Para%20Dummies/00_Codificacion_Variables_Dummies.ipynb), [Contar, decidir y resistir](../../06%20-%20Modelamiento%20Estadistico/Para%20Dummies/01_Modelos_GLM_Robustos_Dummies.ipynb) y [El examen sorpresa](../../06%20-%20Modelamiento%20Estadistico/Para%20Dummies/02_Evaluacion_Modelo_Dummies.ipynb).

In [ ]:
# ============================================================
# Preparación (no modificar): imports y datos inventados
# ============================================================
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

# Reto 1: helados vendidos en 12 días, en cuatro puntos de venta (3 días por punto)
punto = pd.Series(["plaza"] * 3 + ["parque"] * 3 + ["terminal"] * 3 + ["colegio"] * 3)
ventas = np.array([82, 78, 80, 22, 18, 20, 33, 27, 30, 62, 58, 60])
codigo = {"colegio": 0, "parque": 1, "plaza": 2, "terminal": 3}      # numerar en orden alfabético

# Reto 2: temperatura (°C) y ventas de 30 días + 15 columnas de «lotería» (números al azar que no significan nada)
rng = np.random.default_rng(5)
temp = np.round(rng.uniform(15, 30, 30), 1)
vent_d = np.round(20 + 3 * temp + rng.normal(0, 8, 30))
loteria = rng.normal(size=(30, 15))
# Los primeros 20 días son el simulacro (entrenamiento); los últimos 10, el examen sorpresa (prueba)
temp_ent, temp_prb, y_ent, y_prb = temp[:20], temp[20:], vent_d[:20], vent_d[20:]
lot_ent, lot_prb = loteria[:20], loteria[20:]

# Reto 3: grados de calor y helados vendidos en una hora (cuentas pequeñas)
calor = np.arange(18, 31)
conteo = np.array([0, 1, 0, 1, 2, 1, 3, 2, 4, 5, 4, 7, 8])
print("Días de ventas por punto:", len(ventas), "| días de temperatura:", len(temp), "| horas de calor:", len(calor))

---
## Reto 1: El traductor de los puntos de venta 🔤

Doña Marta vende en la **plaza**, el **parque**, el **terminal** y el **colegio**. Para que el modelo entienda el punto, hay dos formas de traducirlo:

- **Numerar** (0, 1, 2, 3 en orden alfabético): el modelo cree que el terminal (3) vale «tres veces» el parque (1)… y que existe una escalera entre los puntos.
- **Casillas de sí/no** (*one-hot*): una columna por punto, con 1 si es ese punto y 0 si no. No hay escalera.

**Pasos**
1. `cod`: el número de cada día según el diccionario `codigo`. Ajusta una `LinearRegression` con `cod` (como columna) y calcula `error_numerado`: el **error absoluto medio** (promedio de |ventas − predicción|).
2. `casillas = pd.get_dummies(punto, dtype=float)`; ajusta otra `LinearRegression` y calcula `error_casillas` (también el error absoluto medio).
3. `n_cols_ref`: cuántas columnas quedan si usas `drop_first=True` (una casilla sobra: se deduce de las demás).

**Puntaje sugerido:** 30 puntos.

In [ ]:
# Casillas que debes llenar (empiezan vacías; no las borres)
cod = error_numerado = casillas = error_casillas = n_cols_ref = None

# 💡 1) Numerar: una sola columna con 0, 1, 2, 3
cod = punto.map(...).to_numpy().reshape(-1, 1)             # busca cada punto en el diccionario `codigo`
modelo_num = LinearRegression().fit(cod, ventas)
error_numerado = np.mean(np.abs(ventas - modelo_num.predict(...)))

# 💡 2) Casillas de sí/no: una columna por punto
casillas = pd.get_dummies(punto, dtype=float)
modelo_cas = LinearRegression().fit(casillas, ...)
error_casillas = np.mean(np.abs(ventas - modelo_cas.predict(casillas)))

# 💡 3) Quitar la casilla que sobra
n_cols_ref = pd.get_dummies(punto, drop_first=..., dtype=float).shape[1]

In [ ]:
# Autoverificación del Reto 1 (no modificar)
assert cod is not None and cod.shape == (12, 1) and list(cod[:4, 0]) == [2, 2, 2, 1], "Pista 🧸: cod = punto.map(codigo).to_numpy().reshape(-1, 1). La plaza vale 2 y el parque 1."
assert error_numerado is not None and abs(error_numerado - 21.0) < 1e-6, "Pista 🧸: ajusta LinearRegression().fit(cod, ventas) y promedia np.abs(ventas - modelo_num.predict(cod))."
assert casillas is not None and casillas.shape == (12, 4) and set(casillas.columns) == {"colegio", "parque", "plaza", "terminal"}, "Pista 🧸: pd.get_dummies(punto, dtype=float) crea una columna por punto (4 en total)."
assert error_casillas is not None and abs(error_casillas - 1.5) < 1e-6, "Pista 🧸: ajusta LinearRegression().fit(casillas, ventas) y promedia el error absoluto de la predicción."
assert error_casillas < error_numerado / 5, "Revisa: con casillas el error debe ser MUCHO menor que numerando (el orden 0-1-2-3 era inventado)."
assert n_cols_ref == 3, "Pista 🧸: con drop_first=True quedan 3 columnas: la cuarta se deduce de las otras tres."
print(f"✅ ¡Reto 1 superado! Numerando los puntos el modelo se equivoca en promedio {error_numerado:.1f} helados por día; con casillas de sí/no, solo {error_casillas:.1f}. Y basta con {n_cols_ref} casillas, porque la cuarta sobra.")

---
## Reto 2: El examen sorpresa del modelo 📝

Doña Marta quiere predecir las ventas con la **temperatura**. Usa los primeros 20 días para que el modelo «estudie» (el simulacro) y los últimos 10 como **examen sorpresa**. Pero alguien le agregó 15 columnas de **lotería**: números al azar sin ningún significado.

**Pasos**
1. Modelo **sencillo** (solo la temperatura): `r2_ent_1` (nota en el simulacro) y `r2_prb_1` (nota en el examen sorpresa), con `modelo.score(X, y)`.
2. Modelo **con lotería** (la temperatura más las 15 columnas de azar): `r2_ent_lot` y `r2_prb_lot`.
3. **Cinco exámenes en lugar de uno:** `r2_cv` es el promedio de `cross_val_score(LinearRegression(), temp.reshape(-1, 1), vent_d, cv=5, scoring="r2")`.

**Puntaje sugerido:** 30 puntos.

In [ ]:
# Casillas que debes llenar
r2_ent_1 = r2_prb_1 = r2_ent_lot = r2_prb_lot = r2_cv = None

# 💡 1) El modelo sencillo: solo la temperatura (una columna)
sencillo = LinearRegression().fit(temp_ent.reshape(-1, 1), y_ent)
r2_ent_1 = sencillo.score(temp_ent.reshape(-1, 1), y_ent)      # nota del simulacro
r2_prb_1 = sencillo.score(temp_prb.reshape(-1, 1), ...)        # nota del examen sorpresa

# 💡 2) El modelo con lotería: temperatura + 15 columnas de azar
X_ent = np.column_stack([temp_ent, lot_ent])
X_prb = np.column_stack([temp_prb, ...])
con_loteria = LinearRegression().fit(X_ent, y_ent)
r2_ent_lot = con_loteria.score(X_ent, y_ent)
r2_prb_lot = con_loteria.score(X_prb, ...)

# 💡 3) Cinco exámenes en lugar de uno
r2_cv = cross_val_score(LinearRegression(), temp.reshape(-1, 1), vent_d, cv=..., scoring="r2").mean()

In [ ]:
# Autoverificación del Reto 2 (no modificar)
_m = LinearRegression().fit(temp_ent.reshape(-1, 1), y_ent)
assert r2_ent_1 is not None and abs(r2_ent_1 - _m.score(temp_ent.reshape(-1, 1), y_ent)) < 1e-9, "Pista 🧸: r2_ent_1 = sencillo.score(temp_ent.reshape(-1, 1), y_ent): la nota del simulacro."
assert r2_prb_1 is not None and abs(r2_prb_1 - _m.score(temp_prb.reshape(-1, 1), y_prb)) < 1e-9, "Pista 🧸: r2_prb_1 usa los datos del examen: sencillo.score(temp_prb.reshape(-1, 1), y_prb)."
_l = LinearRegression().fit(np.column_stack([temp_ent, lot_ent]), y_ent)
assert r2_ent_lot is not None and abs(r2_ent_lot - _l.score(np.column_stack([temp_ent, lot_ent]), y_ent)) < 1e-9, "Pista 🧸: r2_ent_lot = con_loteria.score(X_ent, y_ent)."
assert r2_prb_lot is not None and abs(r2_prb_lot - _l.score(np.column_stack([temp_prb, lot_prb]), y_prb)) < 1e-9, "Pista 🧸: r2_prb_lot = con_loteria.score(X_prb, y_prb), con X_prb = temperatura de prueba + lot_prb."
assert r2_ent_lot > r2_ent_1 + 0.1 and r2_prb_lot < r2_prb_1 - 0.5, "Mira lo que pasó: las columnas de lotería SUBEN la nota del simulacro y HUNDEN la del examen. Revisa que usaste los datos correctos en cada paso."
_cv = cross_val_score(LinearRegression(), temp.reshape(-1, 1), vent_d, cv=5, scoring="r2").mean()
assert r2_cv is not None and abs(r2_cv - _cv) < 1e-9, "Pista 🧸: cross_val_score(LinearRegression(), temp.reshape(-1, 1), vent_d, cv=5, scoring='r2').mean()."
print(f"✅ ¡Reto 2 superado! Con lotería la nota del simulacro sube de {r2_ent_1:.2f} a {r2_ent_lot:.2f}, pero en el examen cae de {r2_prb_1:.2f} a {r2_prb_lot:.2f}. Con cinco exámenes el modelo sencillo saca {r2_cv:.2f} de promedio.")

---
## Reto 3: Contar helados sin números negativos 🔢

Doña Marta anotó cuántos helados vende **en una hora** según los **grados de calor** (`calor`, `conteo`). Son cuentas pequeñas: 0, 1, 2… Una **recta** puede salirse del mundo posible; un modelo hecho para **contar** (Poisson) no.

**Pasos**
1. `pred_recta_10`: lo que predice la recta (`np.polyfit(calor, conteo, 1)`) con **10 grados**, un día frío que nunca vio.
2. Ajusta el modelo de conteos: `glm = sm.GLM(conteo, sm.add_constant(calor), family=sm.families.Poisson()).fit()`; con `b0, b1 = glm.params`, calcula `pred_poisson_10` con `np.exp(b0 + 10 * b1)`.
3. `razon`: `np.exp(b1)`, por cuánto se **multiplican** las ventas con cada grado más de calor.

**Puntaje sugerido:** 25 puntos.

In [ ]:
# Casillas que debes llenar
pred_recta_10 = pred_poisson_10 = razon = None

# 💡 1) La recta (pendiente y corte) y su predicción con 10 grados
pendiente, corte = np.polyfit(calor, conteo, ...)
pred_recta_10 = corte + pendiente * ...

# 💡 2) El modelo de conteos: la predicción siempre es positiva (se usa la exponencial)
glm = sm.GLM(conteo, sm.add_constant(calor), family=sm.families.Poisson()).fit()
b0, b1 = glm.params
pred_poisson_10 = np.exp(b0 + b1 * ...)

# 💡 3) Por cuánto se multiplican las ventas con un grado más
razon = np.exp(...)

In [ ]:
# Autoverificación del Reto 3 (no modificar)
_p, _c = np.polyfit(calor, conteo, 1)
assert pred_recta_10 is not None and abs(pred_recta_10 - (_c + _p * 10)) < 1e-9, "Pista 🧸: np.polyfit(calor, conteo, 1) devuelve (pendiente, corte) de una recta (grado 1); luego corte + pendiente * 10."
assert pred_recta_10 < 0, "Mira: la recta predice helados NEGATIVOS con 10 grados. Si no te sale negativo, revisa el grado del polinomio (debe ser 1)."
_g = sm.GLM(conteo, sm.add_constant(calor), family=sm.families.Poisson()).fit()
assert pred_poisson_10 is not None and abs(pred_poisson_10 - np.exp(_g.params[0] + _g.params[1] * 10)) < 1e-9, "Pista 🧸: pred_poisson_10 = np.exp(b0 + b1 * 10)."
assert 0 < pred_poisson_10 < 1, "El modelo de conteos predice un número positivo y pequeño (menos de un helado por hora) con 10 grados."
assert razon is not None and abs(razon - np.exp(_g.params[1])) < 1e-9 and razon > 1, "Pista 🧸: razon = np.exp(b1). Es mayor que 1: con más calor, las ventas se multiplican."
print(f"✅ ¡Reto 3 superado! Con 10 grados la recta predice {pred_recta_10:.1f} helados (¡imposible!) y el modelo de conteos {pred_poisson_10:.2f}. Cada grado más de calor multiplica las ventas por {razon:.2f}.")

---
## Cierre: ¿qué aprendiste? ✍️

**Puntaje sugerido:** 15 puntos (sin código). Escribe **3 a 5 líneas** en la celda de abajo respondiendo con tus cifras:

- ¿Por qué numerar los puntos de venta (`error_numerado`) hizo que el modelo se equivocara tanto más que las casillas de sí/no (`error_casillas`)?
- ¿Por qué las columnas de lotería subieron la nota del simulacro (`r2_ent_lot`) y hundieron la del examen (`r2_prb_lot`)? ¿Qué te dice `r2_cv` y por qué cinco exámenes son más confiables que uno?
- ¿Por qué la recta predijo `pred_recta_10` helados con 10 grados y por qué el modelo de conteos nunca lo hace?

---
#### ✍️ Tu respuesta

_Escribe aquí tus 3-5 líneas. Reemplaza este texto por tu reflexión._

---
### 🧠 Rúbrica corta

| Criterio | Puntos |
|---|---|
| Reto 1: numerar frente a casillas de sí/no y la casilla que sobra | 30 |
| Reto 2: simulacro frente a examen, columnas de lotería y validación cruzada | 30 |
| Reto 3: recta frente a modelo de conteos | 25 |
| Cierre: reflexión con tus cifras | 15 |
| **Total** | **100** |

**Checklist:** ✅ ejecuté *Run All* sin errores · ✅ no quedan `...` · ✅ las tres autoverificaciones imprimen ✅ · ✅ escribí mi reflexión.

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos (Edición Didáctica Para Dummies)</i>
  </p>
</div>